# Solution: генерация ответов для сдачи

Ноутбук загружает обученную модель и создаёт `answers.jsonl` для бота. На каждую из **1919 задач** генерируется одна программа. Каждая строка результата имеет единственное поле `code`: `{"code": "<вывод модели>"}`.

Все ответы должны генерироваться не дольше **180 секунд на одной NVIDIA T4**. Здесь замеряется вызов `generate_code`, включая токенизацию и декодирование. Загрузка модели, скачивание теста и запись файла выполняются отдельно от замера.

## Пути и параметры

Надо указать три папки в первой ячейке:

- `PROJECT_PATH`: репозиторий с `lib/inference.py`.
- `EXPERIMENT_PATH`: полный путь к папке выбранного эксперимента; его надо задать явно. Это папка с `config.yaml`, `init_state/tokenizer.json` и `checkpoints`.
- `DATA_PATH`: папка с `test.parquet`; если файла нет, тест скачивается с Hugging Face.

`CHECKPOINT` выбирает `best.pt` или `last.pt`. Конфиг модели, токенайзер и веса читаются из одной папки эксперимента. Пути к обучающим данным из сохранённого конфига для загрузки модели не используются. Подходит папка из Kaggle Input или `/kaggle/working/experiment_runs/<имя>` после обучения.

Результат сохраняется в `/kaggle/working/answers.jsonl` на Kaggle или `answers.jsonl` в текущей папке локально. Для установки зависимостей и скачивания теста нужен интернет.

In [ ]:
from pathlib import Path

PROJECT_PATH = Path("/kaggle/input/datasets/wwip62/nlp-hw2-codegen").expanduser().resolve()
EXPERIMENT_PATH = ...  # TODO: полный путь к папке нужного эксперимента
DATA_PATH = Path("/kaggle/input/datasets/wwip62/codegen-tokenized-bpe-16384").expanduser().resolve()
CHECKPOINT = "best"  # "best" или "last"


if EXPERIMENT_PATH is ...:
    raise ValueError("Надо задать EXPERIMENT_PATH: полный путь к папке эксперимента.")
EXPERIMENT_PATH = Path(EXPERIMENT_PATH).expanduser().resolve()

CONFIG_PATH = EXPERIMENT_PATH / "config.yaml"
TOKENIZER_PATH = EXPERIMENT_PATH / "init_state" / "tokenizer.json"
CHECKPOINT_PATH = EXPERIMENT_PATH / "checkpoints" / f"{CHECKPOINT}.pt"

TEST_PATH = DATA_PATH / "test.parquet"
WORK_DIR = (Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()).resolve()
OUTPUT_PATH = WORK_DIR / "answers.jsonl"

In [ ]:
if CHECKPOINT not in ("best", "last"):
    raise ValueError('CHECKPOINT должен быть "best" или "last".')

for name, path in (("PROJECT_PATH", PROJECT_PATH), ("EXPERIMENT_PATH", EXPERIMENT_PATH)):
    if not path.is_dir():
        raise FileNotFoundError(f"{name}: папка не найдена: {path}")

required_files = [
    PROJECT_PATH / "lib" / "inference.py",
    CONFIG_PATH,
    TOKENIZER_PATH,
    CHECKPOINT_PATH,
]
missing_files = [str(path) for path in required_files if not path.is_file()]
if missing_files:
    raise FileNotFoundError("Некорректные пути в первой ячейке. Не найдены файлы:\n" + "\n".join(missing_files))

In [ ]:
print("Репозиторий:", PROJECT_PATH)
print("Эксперимент:", EXPERIMENT_PATH)
print("Конфиг:", CONFIG_PATH)
print("Токенайзер:", TOKENIZER_PATH)
print("Чекпойнт:", CHECKPOINT_PATH)
print("Тест:", TEST_PATH)
print("Результат:", OUTPUT_PATH)

In [4]:
%pip install -q datasets tokenizers tqdm PyYAML "sacrebleu>=2.4,<3" wandb

import importlib
import importlib.util
import json
import os
import subprocess
import sys
from time import perf_counter

os.environ["PROJECT_DIR"] = str(PROJECT_PATH)
sys.path.insert(0, str(PROJECT_PATH))
for name in list(sys.modules):
    if name == "lib" or name.startswith("lib."):
        del sys.modules[name]

Note: you may need to restart the kernel to use updated packages.


In [ ]:
DEVICE = "auto"
MAX_TOKENS = 8196
MAX_NEW_TOKENS = 128
BEAM_TOP_K = 8
VOTING_MODE = "counter"
TIME_LIMIT_SECONDS = 180
EXPECTED_QUESTIONS = 1919

## Импорты

In [ ]:
import pyarrow.parquet as pq
import torch
from huggingface_hub import hf_hub_download

from lib.inference import generate_code, load_for_inference

## Загрузка модели и токенайзера

`load_for_inference` восстанавливает архитектуру из `config.yaml`,
загружает веса выбранного чекпойнта и переводит модель в режим `eval`.

In [ ]:
model, tokenizer = load_for_inference(
    EXPERIMENT_PATH, checkpoint=CHECKPOINT, device=DEVICE,
)
device = next(model.parameters()).device
print(f"Модель: {type(model).__name__} | устройство: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(device)}")

## Чтение тестовых вопросов

Тест читается из `TEST_PATH`. Если локального файла нет, скачивается тестовый Parquet с Hugging Face. Из файла читается только колонка `question`; порядок строк сохраняется.

In [ ]:
if not TEST_PATH.is_file():
    TEST_PATH = hf_hub_download(
        repo_id="SlavaYus/tinysms_qa_12m",
        repo_type="dataset",
        filename="data/test/test.parquet",
    )

questions = pq.read_table(
    TEST_PATH, columns=["question"],
).column("question").to_pylist()
question_source = str(TEST_PATH)


print(f"Источник: {question_source}")
print(f"Прочитано вопросов: {len(questions)}")

## Генерация и замер времени

На CUDA используем FP16 autocast. Внутри `generate_code` вопросы группируются по длине, а ответы возвращаются в исходном порядке.

In [9]:
if device.type == "cuda":
    torch.cuda.synchronize(device)
started = perf_counter()
generated_codes = generate_code(
    model,
    tokenizer,
    questions,
    max_tokens=MAX_TOKENS,
    max_new_tokens=MAX_NEW_TOKENS,
    beam_top_k=BEAM_TOP_K,
    voting_mode=VOTING_MODE,
)
if device.type == "cuda":
    torch.cuda.synchronize(device)
generation_seconds = perf_counter() - started


print(f"Сгенерировано программ: {len(generated_codes)}")
print(f"Время генерации: {generation_seconds:.2f} с")

status = "выполнен" if generation_seconds <= TIME_LIMIT_SECONDS else "превышен"
print(f"Лимит {TIME_LIMIT_SECONDS} с на T4: {status}.")

Сгенерировано программ: 1919
Время генерации: 152.19 с
Лимит 180 с на T4: выполнен.


На всякий случай проверим:

In [10]:
assert len(questions) == len(generated_codes) == EXPECTED_QUESTIONS

## Сохранение в JSONL

Сохраняем вывод модели без изменения пробелов и переносов строк. `json.dumps` экранирует переносы внутри программы, поэтому каждый
JSON-объект занимает ровно одну строку файла.

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with OUTPUT_PATH.open("w", encoding="utf-8") as file:
    for generated_code in generated_codes:
        file.write(json.dumps({"code": generated_code}, ensure_ascii=False) + "\n")

print("Файл:", OUTPUT_PATH)
print("Строк:", len(generated_codes))
print("Пустых ответов:", sum(not generated_code.strip() for generated_code in generated_codes))

Файл: /kaggle/working/answers.jsonl
Строк: 1919
Пустых ответов: 0


/kaggle/working/answers.jsonl